# HellaSwag: `base_100m` vs GPT-2 124M

Same 10,042 validation questions, same scoring code ([evaluation/hellaswag.py](../evaluation/hellaswag.py)) for both models.
Random guessing = 25%. The number usually reported is **acc_norm** (average log-probability per token of each ending).

Runtime: any GPU (an L4 is plenty, ~5-10 min per model). Needs the `HF_TOKEN` secret and the checkpoint on Drive.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
RUN_ROOT = "/content/drive/MyDrive/transformer-from-scratch"
CHECKPOINT = f"{RUN_ROOT}/base_100m/best.pt"

import os
REPO = "/content/transformer-pytorch"
if not os.path.exists(REPO):
    !git clone -q https://github.com/efe-tatar/transformer-pytorch.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q regex tiktoken

from huggingface_hub import hf_hub_download
TOKENIZER = hf_hub_download("Efe-Tatar/fineweb-edu-3b-bpe32k", "tokenizer.json", repo_type="dataset",
                            token=userdata.get("HF_TOKEN"), local_dir="data/fineweb_edu_3b")

## Our model

In [ ]:
!python evaluation/hellaswag.py --checkpoint {CHECKPOINT} --tokenizer {TOKENIZER} --out {RUN_ROOT}/hellaswag_base_100m.json

## GPT-2 124M (the baseline)

In [ ]:
!python evaluation/hellaswag.py --hf gpt2 --out {RUN_ROOT}/hellaswag_gpt2.json

## Side by side

In [ ]:
import json
rows = [json.load(open(f"{RUN_ROOT}/hellaswag_{name}.json")) for name in ("base_100m", "gpt2")]
print(f"{'model':40s} {'acc':>7s} {'acc_norm':>9s}")
for r in rows:
    print(f"{r['model']:40s} {r['acc']:7.2%} {r['acc_norm']:9.2%}")